In [ ]:
!pip uninstall -y numpy scipy pandas grad-cam
!pip install "numpy<2.0" grad-cam scipy pandas shap
!pip install grad-cam

# For Kaggle Notebook
# After this Restart the terminal otherwise code wont work

In [ ]:
import os, glob, time, math, zipfile
import numpy as np
import pandas as pd
import torch
import shap
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.models as models

from skimage.segmentation import slic
from skimage.util import img_as_float

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ============================================================
# CONFIG
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

img_dir = "/kaggle/input/stl10/train_images"

N_IMAGES = 60            # how many folder images to run
SLIC_NSEG = 60
SLIC_COMP = 10.0
SLIC_SIG  = 0.0

NS_CLASSIC = 6000        # classic KernelExplainer nsamples
L1_REG_NUMFEATS = 200    # l1_reg="num_features(200)" (stabilizes regression)

NS_PRIOR = 450           # proposal draws for prior sampler (each draw optionally + complement)
PRIOR_POOL_FRAC = 0.30
PRIOR_PICK_FRAC = 0.60
PAIR_COMPLEMENTS = True

BATCH_SIZE = 64

# ============================================================
# OUTPUT DIRS (CIFAR-style)
# ============================================================
ROOT = "/kaggle/working/artifacts_stl10"
dir_img      = os.path.join(ROOT, "images")
dir_cam      = os.path.join(ROOT, "gradcam")
dir_shap     = os.path.join(ROOT, "classic_superpixel_kernelshap")
dir_priorshp = os.path.join(ROOT, "prior_importance_kernelshap")

for d in [dir_img, dir_cam, dir_shap, dir_priorshp]:
    os.makedirs(d, exist_ok=True)

print("Saving artifacts under:", ROOT)

# ============================================================
# MODEL (ImageNet ResNet-18)
# ============================================================
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights).to(device).eval()

imagenet_mean = np.array(weights.transforms().mean, dtype=np.float32)
imagenet_std  = np.array(weights.transforms().std, dtype=np.float32)

# last conv for Grad-CAM++
target_layer = model.layer4[-1].conv2
cam = GradCAMPlusPlus(model=model, target_layers=[target_layer])

# ============================================================
# HELPERS
# ============================================================
def norm01(a):
    a = a.astype(np.float32)
    mn, mx = float(np.nanmin(a)), float(np.nanmax(a))
    return (a - mn) / (mx - mn + 1e-9)

def save_original(pil_img, out_path):
    pil_img.save(out_path)

def save_overlay(pil_img, heat_2d, out_path, alpha=0.6):
    plt.figure(figsize=(3.5,3.5))
    plt.imshow(pil_img)
    plt.imshow(norm01(heat_2d), alpha=alpha)
    plt.axis("off")
    plt.tight_layout(pad=0)
    plt.savefig(out_path, dpi=200, bbox_inches="tight", pad_inches=0)
    plt.close()

def save_heat_only(heat_2d, out_path):
    plt.figure(figsize=(3.5,3.5))
    plt.imshow(heat_2d, cmap="jet")
    plt.axis("off")
    plt.tight_layout(pad=0)
    plt.savefig(out_path, dpi=200, bbox_inches="tight", pad_inches=0)
    plt.close()

# ------------------ SPEARMAN (no scipy) ------------------
def _rankdata_average_ties(a):
    a = np.asarray(a)
    order = np.argsort(a, kind="mergesort")
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(len(a), dtype=np.float64)
    sorted_a = a[order]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sorted_a[j + 1] == sorted_a[i]:
            j += 1
        if j > i:
            avg = 0.5 * (i + j)
            ranks[order[i:j+1]] = avg
        i = j + 1
    return ranks

def spearman_corr(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    ra = _rankdata_average_ties(a)
    rb = _rankdata_average_ties(b)
    ra -= ra.mean()
    rb -= rb.mean()
    denom = (np.sqrt((ra*ra).sum()) * np.sqrt((rb*rb).sum()) + 1e-12)
    return float((ra*rb).sum() / denom)

# ============================================================
# PRIOR PIPELINE MATH HELPERS
# ============================================================
def _log_comb(n, k):
    return math.lgamma(n + 1) - math.lgamma(k + 1) - math.lgamma(n - k + 1)

def _kernel_weight_log(M, s):
    if s <= 0 or s >= M:
        return -np.inf
    return (math.log(M - 1) - _log_comb(M, s) - math.log(s) - math.log(M - s))

def _size_probs_shapley(M):
    s = np.arange(1, M, dtype=np.float64)
    mass = 1.0 / (s * (M - s))
    mass = mass / mass.sum()
    return mass.astype(np.float64)

def _choose_k_from_prior(rng, s, P, R, prior_pick_frac):
    if P <= 0:
        return 0
    if R <= 0:
        return min(s, P)

    k_min = max(0, s - R)
    k_max = min(s, P)
    if k_min == k_max:
        return int(k_min)

    ks = np.arange(k_min, k_max + 1, dtype=np.int64)
    p = float(prior_pick_frac)

    if p <= 0.0:
        return int(0 if 0 >= k_min and 0 <= k_max else k_min)
    if p >= 1.0:
        return int(k_max)

    logw = np.array([_log_comb(s, int(k)) for k in ks], dtype=np.float64)
    logw = logw + ks * math.log(p) + (s - ks) * math.log(1.0 - p)
    m = np.max(logw)
    w = np.exp(logw - m)
    w = w / w.sum()
    return int(rng.choice(ks, p=w))

def _log_truncbinom_prob(k, s, P, R, prior_pick_frac):
    if P <= 0:
        return 0.0 if k == 0 else -np.inf
    if R <= 0:
        return 0.0 if k == min(s, P) else -np.inf

    k_min = max(0, s - R)
    k_max = min(s, P)
    if k < k_min or k > k_max:
        return -np.inf
    if k_min == k_max:
        return 0.0 if k == k_min else -np.inf

    p = float(prior_pick_frac)
    if p <= 0.0:
        return 0.0 if k == (0 if 0 >= k_min and 0 <= k_max else k_min) else -np.inf
    if p >= 1.0:
        return 0.0 if k == k_max else -np.inf

    ks = np.arange(k_min, k_max + 1, dtype=np.int64)
    logw = np.array([_log_comb(s, int(kk)) for kk in ks], dtype=np.float64)
    logw = logw + ks * math.log(p) + (s - ks) * math.log(1.0 - p)
    m = np.max(logw)
    w = np.exp(logw - m)
    w = w / w.sum()

    idx = int(np.where(ks == k)[0][0])
    return math.log(float(w[idx]) + 1e-300)

# ============================================================
# LIST IMAGES
# ============================================================
exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp")
img_paths = []
for e in exts:
    img_paths.extend(glob.glob(os.path.join(img_dir, e)))
img_paths = sorted(img_paths)

if len(img_paths) == 0:
    raise FileNotFoundError(f"No images found in {img_dir}")

img_paths = img_paths[:min(N_IMAGES, len(img_paths))]
print("Images to run:", len(img_paths))

# ============================================================
# MAIN LOOP
# ============================================================
rows = []

for idx, path in enumerate(img_paths):
    base_name = f"{idx:05d}"
    pil = Image.open(path).convert("RGB")
    img_rgb = np.array(pil).astype(np.float32) / 255.0
    H, W, C = img_rgb.shape

    # normalized HWC
    img_norm = (img_rgb - imagenet_mean[None,None,:]) / imagenet_std[None,None,:]

    # model pred top-1
    with torch.no_grad():
        x0 = torch.from_numpy(img_norm).permute(2,0,1).unsqueeze(0).to(device)
        logits0 = model(x0)[0]
        probs0 = torch.softmax(logits0, dim=0)
        top1_class = int(torch.argmax(probs0).item())
        conf = float(probs0[top1_class].item())

    # SLIC superpixels
    segments = slic(
        img_as_float(img_rgb),
        n_segments=int(SLIC_NSEG),
        compactness=float(SLIC_COMP),
        sigma=float(SLIC_SIG),
        start_label=0,
        channel_axis=-1
    )
    M = int(segments.max() + 1)

    # segmat (M, H*W)
    seg_flat = segments.reshape(-1).astype(np.int64)
    segmat = np.zeros((M, H*W), dtype=np.float32)
    segmat[seg_flat, np.arange(H*W)] = 1.0

    # baseline image in normalized space = all zeros (your “classic” logic)
    base_norm = np.zeros_like(img_norm, dtype=np.float32)
    delta_norm = img_norm - base_norm  # img_norm

    # ------------------------------
    # Save original
    # ------------------------------
    save_original(pil, os.path.join(dir_img, f"{base_name}.png"))

    # ============================================================
    # Grad-CAM++ (for top1)
    # ============================================================
    x0_t = torch.from_numpy(img_norm).permute(2,0,1).unsqueeze(0).to(device)
    targets = [ClassifierOutputTarget(top1_class)]
    cam_map = cam(input_tensor=x0_t, targets=targets, eigen_smooth=True, aug_smooth=True)[0].astype(np.float32)  # (H,W)

    # save GradCAM overlay + heat only
    save_overlay(pil, cam_map, os.path.join(dir_cam, f"{base_name}_cls{top1_class}.png"), alpha=0.6)

    # ============================================================
    # FAST evaluators from masks (shared)
    # ============================================================
    def eval_top1_logit_from_masks(masks_2d, batch_size=BATCH_SIZE):
        masks_2d = np.asarray(masks_2d, dtype=np.float32)
        N = masks_2d.shape[0]
        out = np.empty((N,), dtype=np.float32)

        for s in range(0, N, batch_size):
            e = min(s + batch_size, N)
            z = masks_2d[s:e]  # (b,M)

            on_flat = z @ segmat
            on_map = on_flat.reshape(-1, H, W, 1).astype(np.float32)

            imgs = base_norm[None, ...] + on_map * delta_norm[None, ...]  # = on_map * img_norm
            t = torch.from_numpy(imgs).permute(0,3,1,2).to(device)

            with torch.no_grad():
                logits = model(t)[:, top1_class]
            out[s:e] = logits.detach().cpu().numpy().astype(np.float32)

        return out

    # ============================================================
    # CLASSIC KERNELSHAP (shap.KernelExplainer)
    # ============================================================
    # Features are masks over superpixels
    bg_masks = np.zeros((1, M), dtype=np.float32)
    x_full_mask = np.ones((1, M), dtype=np.float32)

    explainer = shap.KernelExplainer(lambda z: eval_top1_logit_from_masks(z, BATCH_SIZE), bg_masks, link="identity")

    t0 = time.perf_counter()
    phi_super = explainer.shap_values(
        x_full_mask,
        nsamples=NS_CLASSIC,
        l1_reg=f"num_features({int(min(L1_REG_NUMFEATS, M))})"
    )
    t_classic = time.perf_counter() - t0

    # shap may return scalar array or list; normalize to (M,)
    if isinstance(phi_super, list):
        # sometimes list with one element
        phi_super = np.asarray(phi_super[0], dtype=np.float32).reshape(-1)
    else:
        phi_super = np.asarray(phi_super, dtype=np.float32).reshape(-1)

    # Build pixel *density* map (size-normalized) for fair pixel Spearman
    phi_map = np.zeros((H, W), dtype=np.float32)
    for sid in range(M):
        m = (segments == sid)
        cnt = int(m.sum())
        if cnt > 0:
            phi_map[m] = float(phi_super[sid]) / float(cnt)
    classic_phi_pixel = phi_map.reshape(-1)

    # Visual heat (magnitude overlay like your pixel code)
    classic_heat = np.abs(phi_map)
    save_overlay(pil, classic_heat, os.path.join(dir_shap, f"{base_name}_cls{top1_class}.png"), alpha=0.6)

    # ============================================================
    # PRIOR PIPELINE (GradCAM++ prior + importance-corrected WLS)
    # ============================================================
    # score each superpixel by mean CAM
    seg_scores = np.zeros((M,), dtype=np.float32)
    for s in range(M):
        m = (segments == s)
        seg_scores[s] = float(cam_map[m].mean()) if m.any() else -1e9
    seg_ranked = np.argsort(-seg_scores).astype(np.int64)

    pool_size = int(round(M * float(PRIOR_POOL_FRAC)))
    pool_size = max(0, min(M, pool_size))
    prior_pool = seg_ranked[:pool_size]
    rest_pool  = np.setdiff1d(np.arange(M, dtype=np.int64), prior_pool, assume_unique=False)
    P = int(len(prior_pool))
    R = int(len(rest_pool))

    rng = np.random.default_rng(2025 + idx)  # vary seed per image
    size_probs = _size_probs_shapley(M)

    def sample_one_mask_and_logq():
        s_on = int(rng.choice(np.arange(1, M, dtype=np.int64), p=size_probs))
        log_p_size = math.log(float(size_probs[s_on - 1]) + 1e-300)

        if PRIOR_POOL_FRAC <= 0.0 or PRIOR_PICK_FRAC <= 0.0 or P == 0:
            k = 0
        elif R == 0 or PRIOR_PICK_FRAC >= 1.0:
            k = min(s_on, P)
        else:
            k = _choose_k_from_prior(rng, s_on, P, R, PRIOR_PICK_FRAC)

        ids = []
        if k > 0:
            ids.append(rng.choice(prior_pool, size=k, replace=False))
        if (s_on - k) > 0:
            ids.append(rng.choice(rest_pool, size=(s_on - k), replace=False))
        if len(ids) == 0:
            ids = [rng.choice(np.arange(M), size=1, replace=False)]
        ids = np.concatenate(ids, axis=0)

        mask = np.zeros((M,), dtype=np.float32)
        mask[ids] = 1.0

        log_q = log_p_size
        log_q += _log_truncbinom_prob(k, s_on, P, R, PRIOR_PICK_FRAC)
        if k > 0:
            log_q += -_log_comb(P, k)
        if (s_on - k) > 0:
            log_q += -_log_comb(R, s_on - k)

        return mask, s_on, log_q

    # Build Z: baseline + full + samples (+ complements)
    Z0 = np.zeros((1, M), dtype=np.float32)
    Z1 = np.ones((1, M), dtype=np.float32)

    masks = []
    s_list = []
    logq_list = []

    for _ in range(int(NS_PRIOR)):
        m, s_on, logq = sample_one_mask_and_logq()
        masks.append(m); s_list.append(s_on); logq_list.append(logq)
        if PAIR_COMPLEMENTS:
            mc = (1.0 - m).astype(np.float32)
            masks.append(mc); s_list.append(M - s_on); logq_list.append(logq)

    Z_rand = np.stack(masks, axis=0).astype(np.float32)
    Z = np.vstack([Z0, Z1, Z_rand]).astype(np.float32)

    t0 = time.perf_counter()
    y_vals = eval_top1_logit_from_masks(Z, batch_size=BATCH_SIZE)
    t_prior_eval = time.perf_counter() - t0

    f_b = float(y_vals[0])
    f_x = float(y_vals[1])

    # importance-corrected weights
    logw = np.full((Z.shape[0],), -np.inf, dtype=np.float64)
    logw[0] = 20.0
    logw[1] = 20.0
    for irow in range(2, Z.shape[0]):
        s_on = int(s_list[irow - 2])
        lwk = _kernel_weight_log(M, s_on)
        lq  = float(logq_list[irow - 2])
        logw[irow] = lwk - lq

    mlog = np.max(logw[np.isfinite(logw)])
    w = np.exp(logw - mlog).astype(np.float64)
    w[~np.isfinite(w)] = 0.0

    # WLS
    X = np.concatenate([np.ones((Z.shape[0], 1), dtype=np.float64), Z.astype(np.float64)], axis=1)
    Wsqrt = np.sqrt(w).reshape(-1, 1)
    Xw = X * Wsqrt
    yw = y_vals.astype(np.float64) * Wsqrt[:, 0]
    beta, *_ = np.linalg.lstsq(Xw, yw, rcond=None)

    phi_prior_super = beta[1:].astype(np.float32)  # (M,)

    # pixel density map
    phi_map_p = np.zeros((H, W), dtype=np.float32)
    for sid in range(M):
        m = (segments == sid)
        cnt = int(m.sum())
        if cnt > 0:
            phi_map_p[m] = float(phi_prior_super[sid]) / float(cnt)
    prior_phi_pixel = phi_map_p.reshape(-1)

    prior_heat = np.abs(phi_map_p)
    save_overlay(pil, prior_heat, os.path.join(dir_priorshp, f"{base_name}_cls{top1_class}.png"), alpha=0.6)

    # ============================================================
    # Metrics + logging
    # ============================================================
    sp_sp = spearman_corr(phi_super, phi_prior_super)
    sp_px = spearman_corr(classic_phi_pixel, prior_phi_pixel)

    classic_sum_super = float(phi_super.sum())
    prior_sum_super   = float(phi_prior_super.sum())
    classic_sum_pixel = float(classic_phi_pixel.sum())
    prior_sum_pixel   = float(prior_phi_pixel.sum())

    # Additivity sanity on logits:
    # f(x)-f(b) should ~ sum(phi_super). Here baseline feature-mask b=all-zero => masked image = all-zero normalized
    # (We log it, but don’t enforce exact because approximation + l1_reg)
    fx = float(eval_top1_logit_from_masks(np.ones((1,M), dtype=np.float32))[0])
    fb = float(eval_top1_logit_from_masks(np.zeros((1,M), dtype=np.float32))[0])

    rows.append([
        idx,
        os.path.basename(path),
        top1_class,
        conf,
        M,

        t_classic,
        classic_sum_super,
        classic_sum_pixel,
        fx,
        fb,
        (fx - fb),

        t_prior_eval,
        prior_sum_super,
        prior_sum_pixel,
        (f_x - f_b),

        sp_sp,
        sp_px,

        NS_CLASSIC,
        NS_PRIOR,
        PRIOR_POOL_FRAC,
        PRIOR_PICK_FRAC,
        int(PAIR_COMPLEMENTS),
        SLIC_NSEG,
        SLIC_COMP,
        SLIC_SIG
    ])

    print(f"[{idx+1}/{len(img_paths)}] cls={top1_class} conf={conf:.3f} M={M} "
          f"| classic={t_classic:.2f}s prior_eval={t_prior_eval:.2f}s "
          f"| spearman(sp)={sp_sp:.3f} spearman(px)={sp_px:.3f}")

# ============================================================
# SAVE CSV
# ============================================================
df = pd.DataFrame(rows, columns=[
    "ImageId",
    "Filename",
    "Top1Class",
    "Confidence",
    "M_superpixels",

    "Classic_Time_s",
    "Classic_SumPhi_Super",
    "Classic_SumPhi_PixelDensity",
    "Classic_fx",
    "Classic_fb",
    "Classic_fx_minus_fb",

    "Prior_EvalTime_s",
    "Prior_SumPhi_Super",
    "Prior_SumPhi_PixelDensity",
    "Prior_fx_minus_fb",

    "Spearman_Superpixel_Prior_vs_Classic",
    "Spearman_PixelDensity_Prior_vs_Classic",

    "Nsamples_Classic",
    "Nsamples_Prior",
    "PriorPoolFrac",
    "PriorPickFrac",
    "PairComplements",
    "SLIC_NSEG",
    "SLIC_COMP",
    "SLIC_SIG"
])

csv_path = "/kaggle/working/results_stl10.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)
display(df.head())

# ============================================================
# ZIP ARTIFACTS
# ============================================================
zip_path = "/kaggle/working/artifacts_stl10.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(ROOT):
        for fn in files:
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, ROOT)
            zf.write(full, arcname=os.path.join("artifacts_stl10", rel))

print("Zipped artifacts:", zip_path)
print("Done.")
